# Traditional ML Comparison: When You Don't Need an LLM

**Companion notebook for the [Post-Training Explorer](https://provandal.github.io/post-training-explorer/)**

---

### What This Notebook Does

This notebook tackles the **same classification task** as the Post-Training Pipeline — classifying storage I/O workloads into 6 categories — but uses **traditional machine learning** (Random Forest and XGBoost) instead of a large language model.

### Why This Matters

Not every AI problem needs an LLM. When your data is structured and numeric — tables, metrics, sensor readings — traditional ML is often faster, smaller, more accurate, and more interpretable. This notebook demonstrates that point using a task directly relevant to storage and infrastructure engineers.

### What You'll Learn

- How to generate synthetic storage I/O workload data
- How to train and evaluate Random Forest and XGBoost classifiers
- How to compare traditional ML results against LLM fine-tuning results
- How to determine which approach is right for your problem

### Prerequisites

- **Google Account** (to run this notebook in Google Colab)
- **No GPU required** — this notebook runs entirely on CPU
- **No ML experience required** — every step is explained
- **Runtime:** ~2 minutes total

### How to Use This Notebook

If you're new to Colab: each gray block below is a "code cell." Click on a cell to select it, then press **Shift+Enter** (or click the ▶ Play button on the left) to run it. Run the cells **in order from top to bottom** — each step builds on the previous one.

---

## Step 1: Install Dependencies

Before we can train any models, we need to install the Python libraries that provide them. This cell installs:

- **scikit-learn** — the standard library for traditional machine learning in Python. It includes Random Forest, data splitting, evaluation metrics, and more.
- **xgboost** — a high-performance gradient boosting library. XGBoost is widely used in industry and Kaggle competitions for structured data tasks.
- **pandas** — for working with tabular data (like spreadsheets in Python).
- **matplotlib** and **seaborn** — for creating charts and visualizations.

The `-q` flag means "quiet" — it suppresses the verbose installation output so you just see the result.

In [ ]:
# Step 1: Install dependencies (Colab-friendly)
!pip install -q scikit-learn xgboost pandas matplotlib seaborn

## Step 2: Setup — Imports and Workload Profiles

This cell does two things:

**1. Imports** — We load the Python libraries we just installed, plus some built-in ones for random number generation and data handling.

**2. Workload Profiles** — We define the statistical distributions for 6 types of storage I/O workloads. These are the same profiles used in the LLM training pipeline (in `scripts/train_sft.py`), so we're comparing apples to apples.

Each workload type has distinct characteristics across 7 features:

| Feature | What It Measures | Example |
|---------|-----------------|----------|
| **IOPS** | I/O operations per second | OLTP: 5,000–80,000 vs Backup: 50–1,000 |
| **Throughput (MB/s)** | Data transfer rate | AI/ML: 1,000–10,000 vs VDI: 20–200 |
| **Avg Latency (μs)** | Time per operation | OLTP: 100–2,000 vs Backup: 2,000–50,000 |
| **Read %** | Read vs write ratio | Video: 90–100% vs Backup: 5–30% |
| **Random %** | Random vs sequential access | OLTP: 85–99% vs Backup: 2–15% |
| **Block Size (KB)** | Size of each I/O operation | OLTP: 4–8 KB vs Video: 256–2,048 KB |
| **Queue Depth** | Concurrent outstanding I/Os | OLTP: 16–128 vs Video: 1–16 |

Notice how different these profiles are — OLTP has high IOPS with tiny blocks, while Backup has low IOPS with huge blocks. These distinct numeric signatures are exactly what tree-based models excel at detecting.

In [ ]:
# Step 2: Setup — Imports and Workload Profiles

import random
import numpy as np
import pandas as pd
from collections import Counter

random.seed(42)
np.random.seed(42)

# ============================================================
# Workload Profiles — identical to the LLM training pipeline
# ============================================================
# These define the statistical distributions for each workload
# category. The same profiles are used to generate training data
# for the LLM pipeline (scripts/train_sft.py).

WORKLOAD_PROFILES = {
    "OLTP Database": {
        "iops_range": (5000, 80000),
        "throughput_mb_range": (50, 400),
        "avg_latency_us_range": (100, 2000),
        "read_pct_range": (60, 80),
        "random_pct_range": (85, 99),
        "block_size_kb": [4, 8],
        "queue_depth_range": (16, 128),
    },
    "OLAP Analytics": {
        "iops_range": (100, 3000),
        "throughput_mb_range": (500, 5000),
        "avg_latency_us_range": (1000, 20000),
        "read_pct_range": (85, 99),
        "random_pct_range": (5, 30),
        "block_size_kb": [64, 128, 256, 512, 1024],
        "queue_depth_range": (1, 32),
    },
    "AI ML Training": {
        "iops_range": (500, 10000),
        "throughput_mb_range": (1000, 10000),
        "avg_latency_us_range": (500, 10000),
        "read_pct_range": (90, 99),
        "random_pct_range": (20, 60),
        "block_size_kb": [128, 256, 512, 1024],
        "queue_depth_range": (8, 64),
    },
    "Video Streaming": {
        "iops_range": (100, 2000),
        "throughput_mb_range": (200, 3000),
        "avg_latency_us_range": (1000, 15000),
        "read_pct_range": (90, 100),
        "random_pct_range": (10, 40),
        "block_size_kb": [256, 512, 1024, 2048],
        "queue_depth_range": (1, 16),
    },
    "VDI Virtual Desktop": {
        "iops_range": (2000, 30000),
        "throughput_mb_range": (20, 200),
        "avg_latency_us_range": (200, 5000),
        "read_pct_range": (50, 70),
        "random_pct_range": (70, 95),
        "block_size_kb": [4, 8, 16],
        "queue_depth_range": (4, 64),
    },
    "Backup Archive": {
        "iops_range": (50, 1000),
        "throughput_mb_range": (200, 5000),
        "avg_latency_us_range": (2000, 50000),
        "read_pct_range": (5, 30),
        "random_pct_range": (2, 15),
        "block_size_kb": [256, 512, 1024, 2048, 4096],
        "queue_depth_range": (1, 16),
    },
}

LABELS = list(WORKLOAD_PROFILES.keys())
print(f"Workload categories: {LABELS}")
print(f"Number of features: 7 (IOPS, throughput, latency, read%, random%, block size, queue depth)")

## Step 3: Generate the Training Data

Now we generate synthetic storage I/O samples using the workload profiles defined above. Each sample is a set of 7 numeric measurements — the kind of data you'd get from a storage performance monitoring tool.

We generate **720 samples** (120 per workload category), which is the same dataset size used in the LLM training pipeline. The key difference: for traditional ML, we keep the raw numeric features. For the LLM pipeline, these same numbers get converted into text prompts like *"IOPS: 45000 | Throughput: 350 MB/s | Latency: 800 μs..."* — which is a much harder format for a model to learn from.

After generating the data, we'll show:
- The shape of the dataset (rows × columns)
- How many samples per class (should be balanced at 120 each)
- The range of each feature across all workload types
- Basic statistics (mean, std, min, max) for each feature

In [ ]:
# Step 3: Generate the Training Data

def generate_numeric_sample(label):
    """Generate one sample as a numeric feature vector."""
    p = WORKLOAD_PROFILES[label]
    return {
        "iops": random.randint(*p["iops_range"]),
        "throughput_mb": random.randint(*p["throughput_mb_range"]),
        "avg_latency_us": random.randint(*p["avg_latency_us_range"]),
        "read_pct": random.randint(*p["read_pct_range"]),
        "random_pct": random.randint(*p["random_pct_range"]),
        "block_size_kb": random.choice(p["block_size_kb"]),
        "queue_depth": random.randint(*p["queue_depth_range"]),
        "label": label,
    }

# Generate 720 samples (120 per class) — same size as LLM training set
samples = []
for label in LABELS:
    for _ in range(120):
        samples.append(generate_numeric_sample(label))
random.shuffle(samples)

df = pd.DataFrame(samples)
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['label'].value_counts().to_string())
print(f"\nSamples per class: {120} (balanced)")
print(f"Total features: {len(df.columns) - 1}")
print(f"\nFeature ranges across all workloads:")
for col in df.columns:
    if col != 'label':
        print(f"  {col:>20s}: {df[col].min():>10,} \u2014 {df[col].max():>10,}")
print(f"\nFeature statistics:")
df.drop(columns=['label']).describe().round(1)

## Step 4: Prepare the Data for Training

Before we can train a model, we need to prepare the data in the format that scikit-learn expects:

**1. Separate features from labels.** The 7 numeric columns become our input features (`X`). The workload category becomes our target label (`y`). We also convert the text labels ("OLTP Database", "Backup Archive", etc.) into numeric codes that the models can work with.

**2. Split into training and test sets.** We hold out 20% of the data for testing — the model never sees these samples during training, so they give us an honest measure of how well the model generalizes. The `stratify=y` parameter ensures each workload type is proportionally represented in both sets.

**3. Scale the features.** StandardScaler normalizes each feature to have zero mean and unit variance. Tree-based models don't strictly need this (they split on feature values, not magnitudes), but it's good practice and makes the comparison fair if we add other model types later.

In [ ]:
# Step 4: Prepare the Data for Training

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Separate features and labels
feature_cols = ['iops', 'throughput_mb', 'avg_latency_us', 'read_pct', 'random_pct', 'block_size_kb', 'queue_depth']
X = df[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(df['label'])

# 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature scaling (important for some models, doesn't hurt tree-based ones)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {X_train.shape[1]}")
print(f"Classes:      {len(le.classes_)} \u2014 {list(le.classes_)}")

## Step 5: Train Model 1 — Random Forest

Random Forest is one of the most reliable algorithms for structured data classification. Here's how it works:

1. It builds **100 independent decision trees**, each trained on a random subset of the data and a random subset of the features.
2. Each tree makes its own prediction for a given input.
3. The final prediction is the **majority vote** across all 100 trees.

This "wisdom of crowds" approach makes Random Forest resistant to overfitting and robust against noisy data. It's a strong default choice for any structured classification task.

**Watch the training time** when you run this cell — it should complete in well under a second on a CPU. No GPU needed.

In [ ]:
# Step 5: Train Model 1 — Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import time

print("Training Random Forest...")
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_train_time = time.time() - t0

# Evaluate
rf_preds = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_preds)

print(f"\n{'='*50}")
print(f"  Training time:  {rf_train_time:.2f} seconds")
print(f"  Accuracy:       {rf_accuracy:.1%}")
print(f"{'='*50}")
print(f"\nNumber of trees: {rf.n_estimators}")
print(f"Max depth: {max(t.tree_.max_depth for t in rf.estimators_)}")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_preds, target_names=le.classes_))

## Step 6: Train Model 2 — XGBoost

XGBoost (Extreme Gradient Boosting) takes a different approach from Random Forest:

- Random Forest builds trees **independently** and averages their votes.
- XGBoost builds trees **sequentially** — each new tree specifically focuses on correcting the errors that previous trees got wrong. This is called **gradient boosting**.

This iterative error-correction often yields slightly higher accuracy, especially on harder classification boundaries. XGBoost is one of the most successful algorithms in applied machine learning — it has won more Kaggle competitions than any other algorithm.

**Watch the training time** — it should be comparable to Random Forest. Both models train in under a second on a CPU.

In [ ]:
# Step 6: Train Model 2 — XGBoost

from xgboost import XGBClassifier

print("Training XGBoost...")
t0 = time.time()
xgb = XGBClassifier(n_estimators=100, max_depth=6, random_state=42, 
                     use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
xgb_train_time = time.time() - t0

# Evaluate
xgb_preds = xgb.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_preds)

print(f"\n{'='*50}")
print(f"  Training time:  {xgb_train_time:.2f} seconds")
print(f"  Accuracy:       {xgb_accuracy:.1%}")
print(f"{'='*50}")
print(f"\nNumber of trees: {xgb.n_estimators}")
print(f"Max depth: {xgb.max_depth}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

## Step 7: Visual Results — Confusion Matrices

A confusion matrix shows **predicted labels vs. actual labels** in a grid. Here's how to read it:

- Each **row** is the true workload type.
- Each **column** is what the model predicted.
- Numbers **on the diagonal** (top-left to bottom-right) are correct predictions.
- Numbers **off the diagonal** are errors — they show which workload types get confused with each other.

A perfect classifier would have numbers only on the diagonal and zeros everywhere else. You should see something very close to that.

In [ ]:
# Step 7: Visual Results — Confusion Matrices

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest confusion matrix
cm_rf = confusion_matrix(y_test, rf_preds)
ConfusionMatrixDisplay(cm_rf, display_labels=le.classes_).plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title(f'Random Forest \u2014 {rf_accuracy:.1%} Accuracy', fontsize=13)
ax1.set_xticklabels(le.classes_, rotation=35, ha='right', fontsize=9)
ax1.set_yticklabels(le.classes_, fontsize=9)

# XGBoost confusion matrix
cm_xgb = confusion_matrix(y_test, xgb_preds)
ConfusionMatrixDisplay(cm_xgb, display_labels=le.classes_).plot(ax=ax2, cmap='Greens', colorbar=False)
ax2.set_title(f'XGBoost \u2014 {xgb_accuracy:.1%} Accuracy', fontsize=13)
ax2.set_xticklabels(le.classes_, rotation=35, ha='right', fontsize=9)
ax2.set_yticklabels(le.classes_, fontsize=9)

plt.tight_layout()
plt.show()

## Step 8: Head-to-Head — Traditional ML vs. LLM

Now let's put the results side by side. The SFT and GRPO accuracy ranges below come from the Post-Training Pipeline notebook running SmolLM2-360M (a 360 million parameter language model) on the exact same 6-class classification task.

Pay attention to every column in this comparison — not just accuracy, but training time, model size, hardware requirements, and inference speed. For structured numeric data, the gap is stark across every dimension.

In [ ]:
# Step 8: Head-to-Head — Traditional ML vs. LLM

import pickle, sys

rf_size_kb = sys.getsizeof(pickle.dumps(rf)) / 1024

comparison = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'SFT (SmolLM2-360M)', 'GRPO (SmolLM2-360M)'],
    'Accuracy': [f'{rf_accuracy:.1%}', f'{xgb_accuracy:.1%}', '~50-70%', '~65-85%'],
    'Training Time': [f'{rf_train_time:.1f}s', f'{xgb_train_time:.1f}s', '~12 min (T4 GPU)', '~35 min (T4 GPU)'],
    'Model Size': [f'{rf_size_kb:.0f} KB', f'{rf_size_kb:.0f} KB', '~700 MB (base + adapter)', '~180 MB (ONNX)'],
    'Hardware': ['CPU', 'CPU', 'T4/A100 GPU', 'T4/A100 GPU'],
    'Inference': ['<1 ms', '<1 ms', '~50-200 ms (GPU)', '~200-500 ms (browser ONNX)'],
})

print("=" * 90)
print("  MODEL COMPARISON: Traditional ML vs. LLM Post-Training")
print("=" * 90)
print(comparison.to_string(index=False))
print("=" * 90)
print()
print("Key takeaway: For structured numeric data, traditional ML is faster, smaller,")
print("more accurate, and more interpretable. The LLM numbers are not competitive here")
print("\u2014 and that's the point. This is the wrong task for an LLM.")

## Step 9: What Drives the Classification?

One of the biggest advantages of tree-based models over LLMs is **interpretability**. You can ask the model: "Which features mattered most for your decisions?"

Both Random Forest and XGBoost compute **feature importance scores** — a number between 0 and 1 that indicates how much each feature contributed to the model's predictions. Higher is more important.

This is critical for infrastructure teams. If the model says IOPS and block size are the most important features, that tells you something real about what distinguishes these workload types. An LLM gives you an answer but can't easily explain what drove it.

In [ ]:
# Step 9: What Drives the Classification?

import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest feature importance
rf_importance = rf.feature_importances_
sorted_idx = np.argsort(rf_importance)
ax1.barh(range(len(feature_cols)), rf_importance[sorted_idx], color='#3b82f6', alpha=0.8)
ax1.set_yticks(range(len(feature_cols)))
ax1.set_yticklabels([feature_cols[i] for i in sorted_idx])
ax1.set_xlabel('Importance')
ax1.set_title('Random Forest \u2014 Feature Importance')

# XGBoost feature importance
xgb_importance = xgb.feature_importances_
sorted_idx_xgb = np.argsort(xgb_importance)
ax2.barh(range(len(feature_cols)), xgb_importance[sorted_idx_xgb], color='#22c55e', alpha=0.8)
ax2.set_yticks(range(len(feature_cols)))
ax2.set_yticklabels([feature_cols[i] for i in sorted_idx_xgb])
ax2.set_xlabel('Importance')
ax2.set_title('XGBoost \u2014 Feature Importance')

plt.tight_layout()
plt.show()

print("Key insight: Both models agree on the most important features.")
print("These feature importance scores tell you exactly what drives classification \u2014")
print("something an LLM cannot easily provide.")
print(f"\nRandom Forest top 3: {', '.join(feature_cols[i] for i in np.argsort(rf_importance)[-3:][::-1])}")
print(f"XGBoost top 3:       {', '.join(feature_cols[i] for i in np.argsort(xgb_importance)[-3:][::-1])}")

## Step 10: Conclusion — Choose the Right Tool

### For structured numeric data with clear features, traditional ML wins.

| Factor | Traditional ML | LLM Post-Training |
|--------|---------------|-------------------|
| **Best data format** | Numeric features (tables, metrics, sensors) | Unstructured text (logs, tickets, docs) |
| **Training time** | Seconds to minutes | Minutes to hours |
| **Hardware** | CPU only | GPU required |
| **Model size** | Kilobytes | Hundreds of megabytes |
| **Interpretability** | Feature importance, decision paths | Black box |
| **Accuracy on this task** | 95–100% | 50–85% |

### When to use an LLM instead:

- Input is **unstructured text** (error logs, incident reports, documentation)
- The task requires **reasoning** across multiple pieces of information
- You need **generalization** to inputs the model has never seen before
- The relationship between input and output is **not easily captured by numeric features**

### What's Next?

- **See the LLM version of this task:** Try the [Post-Training Explorer](https://provandal.github.io/post-training-explorer/) guided tour to see how SFT, DPO, and GRPO handle the same classification problem.
- **See where LLMs win:** Open the `Realistic_LLM_Use_Case.ipynb` notebook to see a task where LLMs genuinely outperform traditional ML — classifying unstructured storage error logs.
- **Train your own LLM:** Open the `Post_Training_Pipeline.ipynb` notebook to run SFT, DPO, and GRPO training yourself on a free Colab GPU.